In [1]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\ragha\AppData\Local\Temp\ipykernel_33980\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
document=PyPDFLoader("Raghav_Rajaraman_Resume.pdf")
document_content = document.load()[0].page_content
print(document_content)

Raghav R 
Frontend Engineer — Bangalore, India 
+91 6383455639   raghav.rajaraman@gmail.com https://www.linkedin.com/in/raghav-rajaraman-724a7a173/ 
Summary 
Frontend Developer with 4 years of experience developing scalable and user -centric web applications using React.js and 
Next.js. Skilled in building robust frontend architectures, enhancing application performance, and delivering business -critical 
solutions. Actively exploring Generative AI technologies, including LLMs and RAG -based applications, to create smarter and 
more engaging products. 
Education 
VIT Chennai June 2018 – June 2022 
B.Tech - Computer Science and Engineering Chennai, Tamil Nadu 
Experience 
Deloitte India 
Software Engineer - II June 2025 – Present 
• Working on a mission-critical enterprise platform in the  Oil & Gas sector, serving 60M+ users nationally, one of the largest-
scale deployments in the domain.  
• Driving end-to-end feature delivery for a critical application module , owning architecture de

In [3]:
import re

HEADERS = ["SUMMARY", "EDUCATION", "EXPERIENCE", "RESEARCH", "PROJECTS", "TECHNICAL SKILLS"]

regex_exp = r"^(" + "|".join(map(re.escape, HEADERS)) + r")\s*\*?$"
regex_exp

'^(SUMMARY|EDUCATION|EXPERIENCE|RESEARCH|PROJECTS|TECHNICAL\\ SKILLS)\\s*\\*?$'

In [4]:
clean_text = document_content.split("\n")
print(clean_text)

['Raghav R ', 'Frontend Engineer — Bangalore, India ', '+91 6383455639   raghav.rajaraman@gmail.com https://www.linkedin.com/in/raghav-rajaraman-724a7a173/ ', 'Summary ', 'Frontend Developer with 4 years of experience developing scalable and user -centric web applications using React.js and ', 'Next.js. Skilled in building robust frontend architectures, enhancing application performance, and delivering business -critical ', 'solutions. Actively exploring Generative AI technologies, including LLMs and RAG -based applications, to create smarter and ', 'more engaging products. ', 'Education ', 'VIT Chennai June 2018 – June 2022 ', 'B.Tech - Computer Science and Engineering Chennai, Tamil Nadu ', 'Experience ', 'Deloitte India ', 'Software Engineer - II June 2025 – Present ', '• Working on a mission-critical enterprise platform in the  Oil & Gas sector, serving 60M+ users nationally, one of the largest-', 'scale deployments in the domain.  ', '• Driving end-to-end feature delivery for a cr

In [5]:
line = "PROJECTS"

if re.match(regex_exp, line):
    print("Header matched")

Header matched


In [6]:
chunks = {}
content = ""
header = ""
for line in clean_text:
    if re.fullmatch(regex_exp, line.strip().upper()):
        if header:
            chunks[header] = content
        else:
            #For Details Sections as there in no Header for it
            chunks["Contact Information"] = content
        content = ""
        header = line.strip()
        chunks[header] = ""
    else: 
        content = content + line.strip("\n")

#For the last section (Technical Skills)
if(content):
    chunks[header] = content

In [7]:
chunks

{'Contact Information': 'Raghav R Frontend Engineer — Bangalore, India +91 6383455639   raghav.rajaraman@gmail.com https://www.linkedin.com/in/raghav-rajaraman-724a7a173/ ',
 'Summary': 'Frontend Developer with 4 years of experience developing scalable and user -centric web applications using React.js and Next.js. Skilled in building robust frontend architectures, enhancing application performance, and delivering business -critical solutions. Actively exploring Generative AI technologies, including LLMs and RAG -based applications, to create smarter and more engaging products. ',
 'Education': 'VIT Chennai June 2018 – June 2022 B.Tech - Computer Science and Engineering Chennai, Tamil Nadu ',
 'Experience': 'Deloitte India Software Engineer - II June 2025 – Present • Working on a mission-critical enterprise platform in the  Oil & Gas sector, serving 60M+ users nationally, one of the largest-scale deployments in the domain.  • Driving end-to-end feature delivery for a critical applicatio

In [8]:
from langchain_core.documents import Document

docs = []
for chunk in chunks:
    docs.append(Document(metadata={"section": chunk}, page_content=chunks[chunk]))

In [9]:
#Create the embedding model
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
from langchain_community.vectorstores import Chroma

vector_db = Chroma.from_documents(documents=docs, embedding=embedding_model, persist_directory="./resume_db", collection_metadata={"hnsw:space": "cosine"} )

In [11]:
from langchain_community.retrievers import BM25Retriever
import chromadb
bm25_retreiver = BM25Retriever.from_documents(docs)
bm25_retreiver.k = 2
db = Chroma(persist_directory="./resume_db", embedding_function=embedding_model)
chroma_retriever = db.as_retriever(search_kwargs={"k":3})

C:\Users\ragha\AppData\Local\Temp\ipykernel_33980\1549437121.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(persist_directory="./resume_db", embedding_function=embedding_model)


In [12]:
chroma_retriever.invoke("Where did he study?")

[Document(metadata={'section': 'Education'}, page_content='VIT Chennai June 2018 – June 2022 B.Tech - Computer Science and Engineering Chennai, Tamil Nadu '),
 Document(metadata={'section': 'Technical Skills'}, page_content='Programming Languages:  Javascript, Typescript, CSS5, HTML, Solidity , Python Libraries/Frameworks:   React.js, Next.Js, Redux, Context API, Storybook.Js  UI & Styling: Sass, CSS Modules, Storybook, Ant Design, Material UI   Tools & Platforms: Git, GitHub, Figma, Postman , Azure Cloud'),
 Document(metadata={'section': 'Contact Information'}, page_content='Raghav R Frontend Engineer — Bangalore, India +91 6383455639   raghav.rajaraman@gmail.com https://www.linkedin.com/in/raghav-rajaraman-724a7a173/ ')]

In [13]:
query="What are the tech-stack you work on?"
chroma_docs = chroma_retriever.invoke(query)
print(chroma_docs)

[Document(metadata={'section': 'Technical Skills'}, page_content='Programming Languages:  Javascript, Typescript, CSS5, HTML, Solidity , Python Libraries/Frameworks:   React.js, Next.Js, Redux, Context API, Storybook.Js  UI & Styling: Sass, CSS Modules, Storybook, Ant Design, Material UI   Tools & Platforms: Git, GitHub, Figma, Postman , Azure Cloud'), Document(metadata={'section': 'Experience'}, page_content='Deloitte India Software Engineer - II June 2025 – Present • Working on a mission-critical enterprise platform in the  Oil & Gas sector, serving 60M+ users nationally, one of the largest-scale deployments in the domain.  • Driving end-to-end feature delivery for a critical application module , owning architecture decisions, production stability, and mentoring 3 developers  on best practices. • Serve as primary Point of Contact for production incidents & reduced average resolution time by  20% through systematic debugging and cross-functional coordination. • Improved application pe

In [14]:
bm25_docs = bm25_retreiver.invoke(query)
print(bm25_docs)

[Document(metadata={'section': 'Projects'}, page_content='Speedle – Web3 Word Guessing Game  • Built a Wordle-inspired decentralized game using Solidity smart contracts and a React.js frontend, enabling players to participate through Web3 wallets. • Implemented an ETH-based prize pool system where participants contribute funds and daily rewards are automatically distributed to the fastest correct guesser through smart contracts.  Deployed and tested the application on the Sepolia testnet, focusing on secure on -chain game logic & transparent reward distribution '), Document(metadata={'section': 'Experience'}, page_content='Deloitte India Software Engineer - II June 2025 – Present • Working on a mission-critical enterprise platform in the  Oil & Gas sector, serving 60M+ users nationally, one of the largest-scale deployments in the domain.  • Driving end-to-end feature delivery for a critical application module , owning architecture decisions, production stability, and mentoring 3 develo

In [15]:
seen = set()
context_docs = bm25_docs + chroma_docs
unique_docs = [] 

for doc in context_docs:
    if doc.page_content not in seen:
        seen.add(doc.page_content)
        unique_docs.append(doc)
        
unique_docs

[Document(metadata={'section': 'Projects'}, page_content='Speedle – Web3 Word Guessing Game  • Built a Wordle-inspired decentralized game using Solidity smart contracts and a React.js frontend, enabling players to participate through Web3 wallets. • Implemented an ETH-based prize pool system where participants contribute funds and daily rewards are automatically distributed to the fastest correct guesser through smart contracts.  Deployed and tested the application on the Sepolia testnet, focusing on secure on -chain game logic & transparent reward distribution '),
 Document(metadata={'section': 'Experience'}, page_content='Deloitte India Software Engineer - II June 2025 – Present • Working on a mission-critical enterprise platform in the  Oil & Gas sector, serving 60M+ users nationally, one of the largest-scale deployments in the domain.  • Driving end-to-end feature delivery for a critical application module , owning architecture decisions, production stability, and mentoring 3 devel

In [ ]:
RAG_SYSTEM_PROMPT = """You are a professional and friendly AI assistant designed to answer recruitment queries.

You will be provided with retrieved documents containing the candidate's information. Your task is to answer the query based strictly on this context.

Guidelines:
1. Extract and use only the information from the provided context that is directly relevant to the query.
2. Do not assume, extrapolate, or add any external information not explicitly mentioned in the context.
3. If the query is unrelated to the candidate's information, or if the context does not contain the answer, reply exactly with: "I can only answer questions related to the candidate."
4. Do not generate an email if asked. Do not reply anything related to. Leave it as unanswered. You do not have to mention that you do not generate emails
5. Do not answer as 3rd person. Answer in 1st person perspective

Query: {query}

Context: {context_docs}
"""



In [17]:
HUMAN_PROMPT = f"Query: {query} context: {context_docs}"

In [18]:
from utils import gpt_model
from langchain_core.messages import SystemMessage, HumanMessage
result = gpt_model.invoke([SystemMessage(content=RAG_SYSTEM_PROMPT), HumanMessage(content=HUMAN_PROMPT)])

In [19]:
from IPython.display import display, Markdown

display(Markdown(result.content))

The tech stack the candidate works on includes:

- Programming Languages: JavaScript, TypeScript, CSS5, HTML, Solidity, Python
- Libraries/Frameworks: React.js, Next.js, Redux, Context API, Storybook.js
- UI & Styling: Sass, CSS Modules, Storybook, Ant Design, Material UI
- Tools & Platforms: Git, GitHub, Figma, Postman, Azure Cloud
- Blockchain: Solidity (for smart contracts), Web3 (used in a project with Ethereum and Sepolia testnet)
- Areas of exploration: Generative AI, including LLMs and RAG-based applications

In [20]:
#query decomposition function
from langchain.chat_models import init_chat_model
model = init_chat_model(
    "gemma3:1b",
    model_provider="ollama"
)

In [21]:
QUERY_ROUTER = """
You are a query router.

Determine whether the user's query requires query decomposition.

Reply with ONLY one word:
YES
or
NO

YES = The query requires multiple independent retrieval operations.
NO = The query can be answered with a single retrieval operation.

Query:{query}
"""

In [22]:
messages =[SystemMessage(content=QUERY_ROUTER.format(query="What are the educational qualifications and what is the tech stack you have worked on?"))]
messages

[SystemMessage(content="\nYou are a query router.\n\nDetermine whether the user's query requires query decomposition.\n\nReply with ONLY one word:\nYES\nor\nNO\n\nYES = The query requires multiple independent retrieval operations.\nNO = The query can be answered with a single retrieval operation.\n\nQuery:What are the educational qualifications and what is the tech stack you have worked on?\n", additional_kwargs={}, response_metadata={})]

In [23]:
result = model.invoke(messages)
print(result.content)

YES



In [41]:
import asyncio
import time

In [47]:
def get_bm25_docs(query):
    start = time.perf_counter()
    bm25_docs = bm25_retreiver.invoke(query)
    print(f"BM25: {time.perf_counter() - start:.4f}s")
    return bm25_docs

In [48]:
def get_vector_docs(query):
    start = time.perf_counter()
    vector_docs = chroma_retriever.invoke(query)
    print(f"Vector: {time.perf_counter() - start:.4f}s")
    return vector_docs

In [49]:
async def doc_retrieval(query):
    start = time.perf_counter()
    bm25_task = asyncio.to_thread(get_bm25_docs, query)
    vector_task = asyncio.to_thread(get_vector_docs,query)
    
    bm25_result, vector_result = await asyncio.gather(bm25_task, vector_task)
    
    
    end = time.perf_counter()
    
    print(f"Async time: {end - start:.4f} seconds")
    
    return bm25_result + vector_result

In [54]:
await doc_retrieval("What is his skills?")

BM25: 0.0003s
Vector: 2.2254s
Async time: 2.2263 seconds


[Document(metadata={'section': 'Technical Skills'}, page_content='Programming Languages:  Javascript, Typescript, CSS5, HTML, Solidity , Python Libraries/Frameworks:   React.js, Next.Js, Redux, Context API, Storybook.Js  UI & Styling: Sass, CSS Modules, Storybook, Ant Design, Material UI   Tools & Platforms: Git, GitHub, Figma, Postman , Azure Cloud'),
 Document(metadata={'section': 'Projects'}, page_content='Speedle – Web3 Word Guessing Game  • Built a Wordle-inspired decentralized game using Solidity smart contracts and a React.js frontend, enabling players to participate through Web3 wallets. • Implemented an ETH-based prize pool system where participants contribute funds and daily rewards are automatically distributed to the fastest correct guesser through smart contracts.  Deployed and tested the application on the Sepolia testnet, focusing on secure on -chain game logic & transparent reward distribution '),
 Document(metadata={'section': 'Summary'}, page_content='Frontend Develop

In [40]:
for doc in final_docs:
    print(doc)

page_content='Programming Languages:  Javascript, Typescript, CSS5, HTML, Solidity , Python Libraries/Frameworks:   React.js, Next.Js, Redux, Context API, Storybook.Js  UI & Styling: Sass, CSS Modules, Storybook, Ant Design, Material UI   Tools & Platforms: Git, GitHub, Figma, Postman , Azure Cloud' metadata={'section': 'Technical Skills'}
page_content='Speedle – Web3 Word Guessing Game  • Built a Wordle-inspired decentralized game using Solidity smart contracts and a React.js frontend, enabling players to participate through Web3 wallets. • Implemented an ETH-based prize pool system where participants contribute funds and daily rewards are automatically distributed to the fastest correct guesser through smart contracts.  Deployed and tested the application on the Sepolia testnet, focusing on secure on -chain game logic & transparent reward distribution ' metadata={'section': 'Projects'}
page_content='Frontend Developer with 4 years of experience developing scalable and user -centric w

In [51]:
import time

def sync_retrieval(query):
    start = time.perf_counter()

    bm25_result = get_bm25_docs(query)
    vector_result = get_vector_docs(query)

    end = time.perf_counter()

    print(f"Sync time: {end - start:.4f} seconds")

    return bm25_result, vector_result

In [52]:
sync_retrieval("What is his skills")

BM25: 0.0002s
Vector: 0.2306s
Sync time: 0.2311 seconds


([Document(metadata={'section': 'Technical Skills'}, page_content='Programming Languages:  Javascript, Typescript, CSS5, HTML, Solidity , Python Libraries/Frameworks:   React.js, Next.Js, Redux, Context API, Storybook.Js  UI & Styling: Sass, CSS Modules, Storybook, Ant Design, Material UI   Tools & Platforms: Git, GitHub, Figma, Postman , Azure Cloud'),
  Document(metadata={'section': 'Projects'}, page_content='Speedle – Web3 Word Guessing Game  • Built a Wordle-inspired decentralized game using Solidity smart contracts and a React.js frontend, enabling players to participate through Web3 wallets. • Implemented an ETH-based prize pool system where participants contribute funds and daily rewards are automatically distributed to the fastest correct guesser through smart contracts.  Deployed and tested the application on the Sepolia testnet, focusing on secure on -chain game logic & transparent reward distribution ')],
 [Document(metadata={'section': 'Summary'}, page_content='Frontend Dev